In [ ]:
!pip install feedparser
!pip install bs4

try:
  # %tensorflow_version only exists in Colab.
  %tensorflow_version 2.x
except Exception:
  pass
import tensorflow as tf

import numpy as np
import os
import sys
import time
import unicodedata
import re
from sklearn.model_selection import train_test_split
from google.colab import drive

drive.mount('/content/drive')

path='/content/drive/My Drive/Colab Notebooks/competitions/outbrake'
os.chdir(path)

# Converts the unicode file to ascii
def unicode_to_ascii(s):
    return ''.join(c for c in unicodedata.normalize('NFD', s)
        if unicodedata.category(c) != 'Mn')


def preprocess_sentence(w):
    w = unicode_to_ascii(w.lower().strip())

    # replacing everything with space except (0-9, a-z, A-Z, ".", "?", "!", ",")
    w = re.sub(r"[^0-9a-zA-Z?.!,¿]+", " ", w)

    w = w.rstrip().strip()

    # adding a start and an end token to the sentence
    # so that the model know when to start and stop predicting.
    w = '<start> ' + w + ' <end>'
    return w

source = []
target = []
with open("train.txt", "r") as f:
  example = [[],[]]
  for line in f:
    pair = line.split()
    if len(pair) == 2:
      source.append(preprocess_sentence(pair[0]))
      target.append(preprocess_sentence(pair[-1]))

  f.close()

def max_length(tensor):
    return max(len(t) for t in tensor)

def tokenize(lang):
  lang_tokenizer = tf.keras.preprocessing.text.Tokenizer(filters='')
  lang_tokenizer.fit_on_texts(lang)
  tensor = lang_tokenizer.texts_to_sequences(lang)
  tensor = tf.keras.preprocessing.sequence.pad_sequences(tensor, padding='post')
  return tensor, lang_tokenizer

def convert(lang, tensor):
  for t in tensor:
    if t!=0:
      print ("%d ----> %s" % (t, lang.index_word[t]))

class Encoder(tf.keras.Model):
  def __init__(self, vocab_size, embedding_dim, enc_units, batch_sz):
    super(Encoder, self).__init__()
    self.batch_sz = batch_sz
    self.enc_units = enc_units
    self.embedding = tf.keras.layers.Embedding(vocab_size, embedding_dim)
    self.gru = tf.keras.layers.GRU(self.enc_units,
                                   return_sequences=True,
                                   return_state=True,
                                   recurrent_initializer='glorot_uniform')
    self.dropout = tf.keras.layers.Dropout(0.2)

  def call(self, x, hidden, training):
    x = self.embedding(x)
    output, state = self.gru(x, initial_state = hidden)
    output = self.dropout(output, training=training)
    return output, state

  def initialize_hidden_state(self):
    return tf.zeros((self.batch_sz, self.enc_units))

class BahdanauAttention(tf.keras.layers.Layer):
  def __init__(self, units):
    super(BahdanauAttention, self).__init__()
    self.W1 = tf.keras.layers.Dense(units)
    self.W2 = tf.keras.layers.Dense(units)
    self.V = tf.keras.layers.Dense(1)

  def call(self, query, values):
    # hidden shape == (batch_size, hidden size)
    # hidden_with_time_axis shape == (batch_size, 1, hidden size)
    # we are doing this to perform addition to calculate the score
    hidden_with_time_axis = tf.expand_dims(query, 1)

    # score shape == (batch_size, max_length, 1)
    # we get 1 at the last axis because we are applying score to self.V
    # the shape of the tensor before applying self.V is (batch_size, max_length, units)
    score = self.V(tf.nn.tanh(
        self.W1(values) + self.W2(hidden_with_time_axis)))

    # attention_weights shape == (batch_size, max_length, 1)
    attention_weights = tf.nn.softmax(score, axis=1)

    # context_vector shape after sum == (batch_size, hidden_size)
    context_vector = attention_weights * values
    context_vector = tf.reduce_sum(context_vector, axis=1)

    return context_vector, attention_weights

class Decoder(tf.keras.Model):
  def __init__(self, vocab_size, embedding_dim, dec_units, batch_sz):
    super(Decoder, self).__init__()
    self.batch_sz = batch_sz
    self.dec_units = dec_units
    self.embedding = tf.keras.layers.Embedding(vocab_size, embedding_dim)
    self.gru = tf.keras.layers.GRU(self.dec_units,
                                   return_sequences=True,
                                   return_state=True,
                                   recurrent_initializer='glorot_uniform')
    self.fc = tf.keras.layers.Dense(vocab_size)

    # used for attention
    self.attention = BahdanauAttention(self.dec_units)

    self.dropout = tf.keras.layers.Dropout(0.2)

  def call(self, x, hidden, enc_output, training):
    # enc_output shape == (batch_size, max_length, hidden_size)
    context_vector, attention_weights = self.attention(hidden, enc_output)
    context_vector = self.dropout(context_vector, training=training)

    # x shape after passing through embedding == (batch_size, 1, embedding_dim)
    x = self.embedding(x)

    # x shape after concatenation == (batch_size, 1, embedding_dim + hidden_size)
    x = tf.concat([tf.expand_dims(context_vector, 1), x], axis=-1)

    # passing the concatenated vector to the GRU
    output, state = self.gru(x)

    # output shape == (batch_size * 1, hidden_size)
    output = tf.reshape(output, (-1, output.shape[2]))

    # output shape == (batch_size, vocab)
    x = self.fc(output)

    return x, state, attention_weights

input_tensor, inp_lang_tokenizer = tokenize(source)
target_tensor, targ_lang_tokenizer = tokenize(target)

# Calculate max_length of the target tensors
max_length_targ, max_length_inp = max_length(target_tensor), max_length(input_tensor)

# Creating training and validation sets using an 80-20 split
input_tensor_train, input_tensor_val, target_tensor_train, target_tensor_val = train_test_split(input_tensor, target_tensor, test_size=0.2)

BUFFER_SIZE = len(input_tensor_train)
BATCH_SIZE = 8
steps_per_epoch = len(input_tensor_train)//BATCH_SIZE
embedding_dim = 1024
units = 1024
vocab_inp_size = len(inp_lang_tokenizer.word_index) + 1
vocab_tar_size = len(targ_lang_tokenizer.word_index) + 1

dataset = tf.data.Dataset.from_tensor_slices((input_tensor_train, target_tensor_train)).shuffle(BUFFER_SIZE)
dataset = dataset.batch(BATCH_SIZE, drop_remainder=True)

example_input_batch, example_target_batch = next(iter(dataset))
example_input_batch.shape, example_target_batch.shape

encoder = Encoder(vocab_inp_size, embedding_dim, units, BATCH_SIZE)
decoder = Decoder(vocab_tar_size, embedding_dim, units, BATCH_SIZE)
optimizer = tf.keras.optimizers.Adam()
loss_object = tf.keras.losses.SparseCategoricalCrossentropy(
    from_logits=True, reduction='none')

def loss_function(real, pred):
  mask = tf.math.logical_not(tf.math.equal(real, 0))
  loss_ = loss_object(real, pred)
  mask = tf.cast(mask, dtype=loss_.dtype)
  loss_ = tf.multiply(loss_, mask)
  return tf.reduce_mean(loss_)

@tf.function
def train_step(inp, targ, enc_hidden):
  loss = 0

  with tf.GradientTape() as tape:
    enc_output, enc_hidden = encoder(inp, enc_hidden, True)
    dec_hidden = enc_hidden
    dec_input = tf.expand_dims([targ_lang_tokenizer.word_index['<start>']] * BATCH_SIZE, 1)

    # Teacher forcing - feeding the target as the next input
    for t in range(1, targ.shape[1]):
      # passing enc_output to the decoder
      predictions, dec_hidden, _ = decoder(dec_input, dec_hidden, enc_output, True)

      loss += loss_function(targ[:, t], predictions)

      # using teacher forcing
      dec_input = tf.expand_dims(targ[:, t], 1)

  batch_loss = (loss / int(targ.shape[1]))
  variables = encoder.trainable_variables + decoder.trainable_variables
  gradients = tape.gradient(loss, variables)
  optimizer.apply_gradients(zip(gradients, variables))
  return batch_loss

def evaluate(sentence):
    attention_plot = np.zeros((max_length_targ, max_length_inp))
    sentence = preprocess_sentence(sentence)

    inputs = [inp_lang_tokenizer.word_index.get(i, 0) for i in sentence.split(' ')]
    inputs = tf.keras.preprocessing.sequence.pad_sequences([inputs],
                                                           maxlen=max_length_inp,
                                                           padding='post')
    inputs = tf.convert_to_tensor(inputs)

    result = ''
    hidden = [tf.zeros((1, units))]
    enc_out, enc_hidden = encoder(inputs, hidden, False)

    dec_hidden = enc_hidden
    dec_input = tf.expand_dims([targ_lang_tokenizer.word_index['<start>']], 0)

    result ='<start> '
    for t in range(max_length_targ):
        predictions, dec_hidden, attention_weights = decoder(dec_input,
                                                             dec_hidden,
                                                             enc_out,
                                                             False)

        # storing the attention weights to plot later on
        attention_weights = tf.reshape(attention_weights, (-1, ))
        attention_plot[t] = attention_weights.numpy()

        predicted_id = tf.argmax(predictions[0]).numpy()
        result += targ_lang_tokenizer.index_word[predicted_id] + ' '

        if targ_lang_tokenizer.index_word[predicted_id] == '<end>':
            return result, sentence, attention_plot

        # the predicted ID is fed back into the model
        dec_input = tf.expand_dims([predicted_id], 0)
    return result, preprocessed, attention_plot

EPOCHS = 32
epoch=-1
total_loss=1
total_loss_cut=0.001*steps_per_epoch*BATCH_SIZE
training_start_time=time.time()

# early stopping parameters
best_total_loss = sys.maxsize
patience = 16
early_stopping = False
stopping_step = 0

print("\nStarting training of at most {} epochs or if no improvement after {} epochs".format(EPOCHS,patience))
while (epoch < EPOCHS) and (early_stopping == False):
  epoch+=1

  start = time.time()
  enc_hidden = encoder.initialize_hidden_state()
  total_loss = 0

  for (batch, (inp, targ)) in enumerate(dataset.take(steps_per_epoch)):
    batch_loss = train_step(inp, targ, enc_hidden)
    total_loss += batch_loss

    if batch%8 == 0:
      print('Epoch {} Batch {} Loss {:.4f}'.format(epoch + 1,
                                                     batch,
                                                     batch_loss.numpy()))
  '''
  # saving (checkpoint) the model every 2 epochs
  if (epoch + 1) % 2 == 0:
    checkpoint.save(file_prefix = checkpoint_prefix)
  '''

  # early stopping
  if (total_loss < best_total_loss):
    stopping_step = 0
    best_total_loss = total_loss
  else:
    stopping_step += 1
  if stopping_step >= patience:
    early_stopping = True

  print('Epoch {} Total Loss {:.4f}'.format(epoch + 1, total_loss))
  print('Time taken for this epoch {:.4f} sec\n'.format(time.time() - start))

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.3/81.3 kB 973.5 kB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for sgmllib3k: filename=sgmllib3k-1.0.0-py3-none-any.whl size=6048 sha256=dc130184211f5fa01653d4c8c23ff73aa3861c2a990834c2cd6403f6c9ff60ea
  Stored in directory: /root/.cache/pip/wheels/f0/69/93/a47e9d621be168e9e33c7ce60524393c0b92ae83cf6c6e89c5
Successfully built sgmllib3k
Colab only includes TensorFlow 2.x; %tensorflow_version has no effect.
Mounted at /content/drive

Starting training of at most 32 epochs or if no improvement after 16 epochs
Epoch 1 Batch 0 Loss 1.2864
Epoch 1 Batch 8 Loss 0.0697
Epoch 1 Batch 16 Loss 0.2306
Epoch 1 Batch 24 Loss 0.3021
Epoch 1 Batch 32 Loss 0.2712
Epoch 1 Batch 40 Loss 0.3861
Epoch 1 Batch 48 Loss 0.3049
Epoch 1 Batch 56 Loss 0.2014
Epoch 1 Batch 64 Loss 0.2499
Epoch 1 Batch 72 Loss 0.3225
Epoch 1 Batch 80 Loss 0.2062
Epoch 1 Batch 88 Loss 0.1781
Epoch 1 Batch 96 Loss 0.2618
Epoch 1 Batch 104 Loss 0.2216
Epoch 1

In [ ]:
def resolve(word):
    tag, preprocessed, attention_plot = evaluate(word)
    tag = tag.replace("<start>", "")
    tag = tag.replace("<end>", "")
    tag = tag.strip()
    return tag

import feedparser
from bs4 import BeautifulSoup

friendly_label = {
    "case": "Case: ",
    "date": "Date: ",
    "disease": "Disease: ",
    "loc": "Location: "
}

feed = feedparser.parse("https://www.who.int/feeds/entity/csr/don/en/rss.xml")
entries = feed.entries
for entry in entries:
  word_tags = []
  soup = BeautifulSoup(entry.title, features="html.parser")
  title = soup.get_text()
  print(title)
  print(entry.link)
  print(entry.summary)

  tokens = title.split()
  for token in tokens:
    tag = resolve(token)
    if tag != 'o':
      word_tags.append(tuple([token, tag]))

  soup = BeautifulSoup(entry.description, features="html.parser")
  description = soup.get_text()
  tokens = description.split()
  for token in tokens:
    tag = resolve(token)
    if tag != 'o':
      word_tags.append(tuple([token, tag]))

  entities = {}
  for word_tag in word_tags:
    wt = word_tag[1].split()
    triple = (word_tag[0], wt[0], wt[1])
    section = triple[1]
    entity = triple[2]

    entity_list = entities.get(entity, [])

    if section == 'b':
      entity_list.append([triple])
    elif section == 'i':
      if entity_list:
          last_entity = entity_list[-1]
          last_entity.append(triple)
      else:
          entity_list.append([triple])
    entities[entity] = entity_list

  for key, items in entities.items():
    tuple_list = []
    for item in items:
      tuple_list.append(tuple(item))

    unique_list = list(dict.fromkeys(tuple_list))
    for u in unique_list:
      print (friendly_label[key] + " ".join([t[0] for t in list(u)]))

  print()